# SentenceChunker Demo
Shows character-mode and token-mode sentence-boundary chunking on a plain-text fixture.

## Imports

In [ ]:
# Standard Library
from pathlib import Path

# Third Party Library

# Private Library
from cleave.chunker.sentence import SentenceChunker
from cleave.parsers.factory import ParserFactory
from cleave.schemas import (
    ChunkParams, ChunkUnit,
    ContentBlock, ContentType,
    Document, DocumentPage, Source, SourceType,
)

## Fixture

In [ ]:
from tests.fixtures.txt import create_sample_txt

FIXTURE_PATH = Path("tests/fixtures/sample.txt")
if not FIXTURE_PATH.exists():
    create_sample_txt(FIXTURE_PATH)

print(f"Fixture: {FIXTURE_PATH}")
print(FIXTURE_PATH.read_text(encoding="utf-8"))

In [ ]:
doc_parser = ParserFactory.create(str(FIXTURE_PATH))
doc = doc_parser.parse()

print(f"Pages : {len(doc.pages)}")
print(f"Chars : {sum(len(b.content) for p in doc.pages for b in p.blocks)}")

## Character mode

In [ ]:
CHUNK_SIZE = 150
CHUNK_OVERLAP = 20

char_chunker = SentenceChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP))
char_chunks = char_chunker.chunk(doc)

print(f"chunk_size={CHUNK_SIZE}  chunk_overlap={CHUNK_OVERLAP}")
print(f"Total chunks : {len(char_chunks)}\n")
for c in char_chunks:
    print(f"[{c.index}] chars {c.char_start:>3}\u2013{c.char_end:<3}  tokens={c.token_count:>3}  \u2502 {c.text[:60]!r}")

### Overlap inspection
The last `chunk_overlap` characters of chunk *n* should appear at the start of chunk *n+1*.

In [ ]:
for a, b in zip(char_chunks, char_chunks[1:]):
    tail = a.text[-CHUNK_OVERLAP:]
    head = b.text[:CHUNK_OVERLAP]
    match = "\u2713" if tail == head else "\u2717"
    print(f"chunk {a.index}\u2192{b.index}  {match}  overlap: {tail!r}")

## Token mode

In [ ]:
TOKEN_SIZE = 30
TOKEN_OVERLAP = 6

tok_chunker = SentenceChunker(
    ChunkParams(chunk_size=TOKEN_SIZE, chunk_overlap=TOKEN_OVERLAP, unit=ChunkUnit.tokens)
)
tok_chunks = tok_chunker.chunk(doc)

print(f"chunk_size={TOKEN_SIZE} tokens  chunk_overlap={TOKEN_OVERLAP} tokens")
print(f"Total chunks : {len(tok_chunks)}\n")
for c in tok_chunks:
    print(f"[{c.index}] chars {c.char_start:>3}\u2013{c.char_end:<3}  tokens={c.token_count:>3}  \u2502 {c.text[:60]!r}")

## Character vs Token Comparison

In [ ]:
print(f"{'Mode':<12} {'Chunks':>6}  {'Avg chars':>10}  {'Avg tokens':>10}")
print("-" * 44)

for label, chunks in [("characters", char_chunks), ("tokens", tok_chunks)]:
    avg_chars = sum(len(c.text) for c in chunks) / len(chunks)
    avg_tokens = sum(c.token_count for c in chunks) / len(chunks)
    print(f"{label:<12} {len(chunks):>6}  {avg_chars:>10.1f}  {avg_tokens:>10.1f}")

## Mixed Content

Text blocks are split at sentence boundaries; table and image blocks are emitted as atomic chunks.

In [ ]:
mixed_source = Source(source_type=SourceType.txt, name="demo.txt", location="demo.txt")
mixed_doc = Document(
    source=mixed_source,
    pages=[
        DocumentPage(
            page_number=1,
            blocks=[
                ContentBlock(type=ContentType.text,  content="Introduction text before the table. It ends here.", position=0),
                ContentBlock(type=ContentType.table, content="| Col A | Col B |\n|--------|--------|\n| foo    | bar    |\n| baz    | qux    |", position=1),
                ContentBlock(type=ContentType.image, content="<base64-image-data>", position=2),
                ContentBlock(type=ContentType.text,  content="Summary text after the image. It also ends here.", position=3),
            ],
        )
    ],
    total_pages=1,
)

mixed_chunker = SentenceChunker(ChunkParams(chunk_size=80, chunk_overlap=10))
mixed_chunks = mixed_chunker.chunk(mixed_doc)

print(f"Total chunks: {len(mixed_chunks)}\n")
for c in mixed_chunks:
    print(f"[{c.index}] type={c.content_type.value:<6}  \u2502 {c.text[:60]!r}")